In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os, csv, json, random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter, defaultdict
import copy
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
 
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
ZIP1 = os.path.join(DATASET_DIR, "Hi-EF-20260829T071606Z-1-001", "Hi-EF")
 
annotations = {}
with open(os.path.join(ZIP1, "annotation.csv"), 'r') as f:
    for row in csv.reader(f):
        if len(row) >= 2:
            cid = row[0].strip()
            annotations[cid] = {
                'emotion': row[7].strip() if len(row) > 7 and row[7].strip() else None,
                'polarity': row[5].strip() if len(row) > 5 and row[5].strip() else None,
                'intensity': row[6].strip() if len(row) > 6 and row[6].strip() else None,
            }
 
samples = []
with open(os.path.join(ZIP1, "sample.csv"), 'r') as f:
    reader = csv.reader(f); next(reader)
    for row in reader: samples.append(row)
 
emotion_map = {'angry':0,'disgust':1,'fear':2,'happy':3,'neutral':4,'sad':5,'surprise':6}
polarity_map = {'positive':0,'neutral':1,'negative':2}
intensity_map = {'weak':0,'powerful':1}
emo_names = ['angry','disgust','fear','happy','neutral','sad','surprise']
 
mcis_index = []
for s in samples:
    clips = [s[i].strip() for i in range(1, 5)]
    entry = {'sample_id': s[0].strip(), 'clip_ids': clips,
             'feature_files': [c.replace('/','_')+'.pt' for c in clips]}
    for ln, ci in [('clip3',2),('clip4',3)]:
        cid = clips[ci]
        if cid in annotations and annotations[cid].get('emotion'):
            ann = annotations[cid]
            entry[f'{ln}_emotion'] = emotion_map.get(ann['emotion'],-1)
            entry[f'{ln}_polarity'] = polarity_map.get(ann.get('polarity',''),-1)
            entry[f'{ln}_intensity'] = intensity_map.get(ann.get('intensity',''),-1)
        else:
            entry[f'{ln}_emotion']=-1; entry[f'{ln}_polarity']=-1; entry[f'{ln}_intensity']=-1
    mcis_index.append(entry)
 
def split_mcis(mcis_index, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    clip4_groups = {}
    for idx, e in enumerate(mcis_index):
        c4 = e['clip_ids'][3]
        clip4_groups.setdefault(c4, []).append(idx)
    keys = list(clip4_groups.keys()); rng.shuffle(keys)
    n = len(keys); nt = int(n*train_ratio); nv = int(n*val_ratio)
    return ([i for g in keys[:nt] for i in clip4_groups[g]],
            [i for g in keys[nt:nt+nv] for i in clip4_groups[g]],
            [i for g in keys[nt+nv:] for i in clip4_groups[g]])
 
train_idx, val_idx, test_idx = split_mcis(mcis_index)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

Device: cuda
Train: 1980, Val: 424, Test: 426


In [3]:
class HiEFDataset(Dataset):
    """Standard dataset. shuffle_a=True randomly permutes clip III across samples."""
    def __init__(self, mcis_index, features_dir, indices, shuffle_a=False, seed=42):
        self.mcis_index = mcis_index
        self.features_dir = features_dir
        self.indices = indices
        self.shuffle_a = shuffle_a
        if shuffle_a:
            rng = random.Random(seed)
            self.shuffled_map = list(range(len(indices)))
            rng.shuffle(self.shuffled_map)
    
    def __len__(self): return len(self.indices)
    
    def _load(self, ff):
        d = torch.load(os.path.join(self.features_dir, ff), map_location='cpu', weights_only=False)
        return d['face_features'], d['ori_features'], d['text_feature'], d.get('audio_feature', torch.zeros(527))
    
    def __getitem__(self, idx):
        e = self.mcis_index[self.indices[idx]]
        c1f,c1o,c1t,c1a = self._load(e['feature_files'][0])
        c2f,c2o,c2t,c2a = self._load(e['feature_files'][1])
        
        # Clip III: use shuffled index if shuffle_a=True
        if self.shuffle_a:
            e_a = self.mcis_index[self.indices[self.shuffled_map[idx]]]
            c3f,c3o,c3t,c3a = self._load(e_a['feature_files'][2])
            clip3_emo = e_a['clip3_emotion']
        else:
            c3f,c3o,c3t,c3a = self._load(e['feature_files'][2])
            clip3_emo = e['clip3_emotion']
        
        return {
            'clip1_face':c1f,'clip1_ori':c1o,'clip1_text':c1t,'clip1_audio':c1a,
            'clip2_face':c2f,'clip2_ori':c2o,'clip2_text':c2t,'clip2_audio':c2a,
            'clip3_face':c3f,'clip3_ori':c3o,'clip3_text':c3t,'clip3_audio':c3a,
            'target':e['clip4_emotion'],
            'clip3_emotion':clip3_emo,
            'clip4_polarity':e['clip4_polarity'],
            'clip4_intensity':e['clip4_intensity'],
        }
 
def collate_fn(batch):
    r = {}
    for k in [f'clip{c}_{m}' for c in [1,2,3] for m in ['face','ori','text','audio']]:
        r[k] = torch.stack([b[k] for b in batch])
    for k in ['target','clip3_emotion','clip4_polarity','clip4_intensity']:
        r[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
    return r
 
class TemporalTransformer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, 16, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, 8, d*4, 0.1, batch_first=True, norm_first=True)
        self.t = nn.TransformerEncoder(el, num_layers=2)
    def forward(self, x): return self.t(x + self.pos[:, :x.size(1), :])
 
class CrossAttentionFusion(nn.Module):
    def __init__(self, d=512, n_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([nn.MultiheadAttention(d, 8, dropout=0.1, batch_first=True) for _ in range(n_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(n_layers)])
    def forward(self, q, kv):
        x = q
        for attn, norm in zip(self.layers, self.norms):
            o, _ = attn(x, kv, kv); x = norm(x + o)
        return x
 
class ClipEncoder(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.face_t = TemporalTransformer(d)
        self.ori_t = TemporalTransformer(d)
        self.type_f = CrossAttentionFusion(d)
        self.audio_proj = nn.Linear(527, d)
        self.mod_f = CrossAttentionFusion(d)
    def forward(self, face, ori, text, audio):
        f = self.face_t(face).mean(1, keepdim=True)
        o = self.ori_t(ori).mean(1, keepdim=True)
        v = self.type_f(f, torch.cat([f, o], 1))
        a = self.audio_proj(F.normalize(audio, dim=-1)).unsqueeze(1)
        t = text.unsqueeze(1)
        return self.mod_f(v, torch.cat([v, t, a], 1)).squeeze(1)
 
def make_head(d=512, n=7):
    return nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d,d//2),
                         nn.GELU(), nn.Dropout(0.2), nn.Linear(d//2, n))
 
# %%
# --- Models ---
 
class M1_Context(nn.Module):
    """P(B|C): clips I+II only."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.ctx_f = CrossAttentionFusion(d)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        fused = self.ctx_f(c1.unsqueeze(1), torch.stack([c1,c2],1)).squeeze(1)
        return self.head(fused)
 
class M3_Full(nn.Module):
    """P(B|C,A): clips I+II+III."""
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.lstm = nn.LSTM(d, d, num_layers=3, batch_first=False, dropout=0.1)
        self.pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        el = nn.TransformerEncoderLayer(d, 8, d*4, 0.1, batch_first=True, norm_first=True)
        self.trans = nn.TransformerEncoder(el, num_layers=2)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        c3 = self.enc(batch['clip3_face'], batch['clip3_ori'], batch['clip3_text'], batch['clip3_audio'])
        seq = torch.stack([c1,c2,c3], 0)
        out, _ = self.lstm(seq)
        return self.head(self.trans(out.permute(1,0,2) + self.pos).mean(1))
 
class M_EmoLabel(nn.Module):
    """P(B|C, E_A): context + A's emotion as 7-dim one-hot (not full features)."""
    def __init__(self, d=512, n_emo=7):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.ctx_f = CrossAttentionFusion(d)
        self.emo_embed = nn.Embedding(n_emo, d)
        self.combine = CrossAttentionFusion(d, n_layers=2)
        self.head = make_head(d)
    def forward(self, batch):
        c1 = self.enc(batch['clip1_face'], batch['clip1_ori'], batch['clip1_text'], batch['clip1_audio'])
        c2 = self.enc(batch['clip2_face'], batch['clip2_ori'], batch['clip2_text'], batch['clip2_audio'])
        ctx = self.ctx_f(c1.unsqueeze(1), torch.stack([c1,c2],1)).squeeze(1)
        
        emo_a = self.emo_embed(batch['clip3_emotion'].clamp(min=0))  # [B, 512]
        
        ctx_q = ctx.unsqueeze(1)
        emo_q = emo_a.unsqueeze(1)
        combined = self.combine(emo_q, torch.cat([emo_q, ctx_q], 1)).squeeze(1)
        return self.head(combined)

In [4]:
BATCH_SIZE = 32
 
def compute_metrics(preds, labels, n_classes=7):
    preds, labels = np.array(preds), np.array(labels)
    war = (preds == labels).sum() / len(labels) * 100
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() > 0: recalls.append((preds[mask] == c).sum() / mask.sum() * 100)
    return war, np.mean(recalls) if recalls else 0.0
 
def train_and_eval(model, train_loader, val_loader, test_loader, n_epochs=30, lr=1e-4, name="m"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    best_uar = 0
    for ep in range(1, n_epochs+1):
        model.train()
        for b in train_loader:
            bg = {k:v.to(DEVICE) for k,v in b.items()}
            loss = F.cross_entropy(model(bg), bg['target'])
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        model.eval(); vp,vl=[],[]; vl_loss=0
        with torch.no_grad():
            for b in val_loader:
                bg = {k:v.to(DEVICE) for k,v in b.items()}
                lo = model(bg)
                vl_loss += F.cross_entropy(lo, bg['target']).item()*lo.size(0)
                vp.extend(lo.argmax(-1).cpu().numpy()); vl.extend(b['target'].numpy())
        scheduler.step(vl_loss/len(val_loader.dataset))
        _,vu = compute_metrics(vp,vl)
        if vu > best_uar: best_uar=vu; torch.save(model.state_dict(), f'/kaggle/working/{name}.pt')
    
    model.load_state_dict(torch.load(f'/kaggle/working/{name}.pt', map_location=DEVICE, weights_only=True))
    model.eval(); tp,tl,t_pol,t_int=[],[],[],[]
    with torch.no_grad():
        for b in test_loader:
            bg = {k:v.to(DEVICE) for k,v in b.items()}
            tp.extend(model(bg).argmax(-1).cpu().numpy()); tl.extend(b['target'].numpy())
            t_pol.extend(b['clip4_polarity'].numpy()); t_int.extend(b['clip4_intensity'].numpy())
    tw,tu = compute_metrics(tp,tl)
    return {'war':tw,'uar':tu,'preds':np.array(tp),'labels':np.array(tl),
            'pol':np.array(t_pol),'int':np.array(t_int)}
 
def bootstrap_ci(preds, labels, n_boot=1000, ci=95):
    """Bootstrap confidence interval for UAR."""
    rng = np.random.RandomState(42)
    uars = []
    for _ in range(n_boot):
        idx = rng.choice(len(labels), len(labels), replace=True)
        _, u = compute_metrics(preds[idx], labels[idx])
        uars.append(u)
    lo = np.percentile(uars, (100-ci)/2)
    hi = np.percentile(uars, 100-(100-ci)/2)
    return np.mean(uars), lo, hi
 
def bootstrap_delta_ci(preds1, preds2, labels, n_boot=1000, ci=95):
    """Bootstrap CI for Δ_UAR = UAR(model2) - UAR(model1)."""
    rng = np.random.RandomState(42)
    deltas = []
    for _ in range(n_boot):
        idx = rng.choice(len(labels), len(labels), replace=True)
        _, u1 = compute_metrics(preds1[idx], labels[idx])
        _, u2 = compute_metrics(preds2[idx], labels[idx])
        deltas.append(u2 - u1)
    lo = np.percentile(deltas, (100-ci)/2)
    hi = np.percentile(deltas, 100-(100-ci)/2)
    mean = np.mean(deltas)
    p_positive = np.mean(np.array(deltas) > 0)
    return mean, lo, hi, p_positive

In [5]:
print("=" * 70)
print("EXP 1: Multiple Seeds (3 seeds × M1, M3)")
print("=" * 70)
 
SEEDS = [42, 123, 456]
N_EPOCHS = 30
 
multi_results = {'M1': [], 'M3': []}
 
for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    
    # Standard loaders (no shuffle_a)
    trl = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, train_idx, seed=seed), 
                     batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, val_idx),
                     batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    tst = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, test_idx),
                     batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
    
    print(f"  M1 (context only)...", end=" ", flush=True)
    m1 = M1_Context().to(DEVICE)
    r1 = train_and_eval(m1, trl, val, tst, N_EPOCHS, name=f"m1_s{seed}")
    multi_results['M1'].append(r1)
    print(f"WAR={r1['war']:.1f}%, UAR={r1['uar']:.1f}%")
    
    print(f"  M3 (full)...", end=" ", flush=True)
    m3 = M3_Full().to(DEVICE)
    r3 = train_and_eval(m3, trl, val, tst, N_EPOCHS, name=f"m3_s{seed}")
    multi_results['M3'].append(r3)
    print(f"WAR={r3['war']:.1f}%, UAR={r3['uar']:.1f}%")
 
# %%
# Summary with CI
print("\n--- Multi-seed summary ---")
for model_name in ['M1', 'M3']:
    uars = [r['uar'] for r in multi_results[model_name]]
    wars = [r['war'] for r in multi_results[model_name]]
    print(f"{model_name}: WAR={np.mean(wars):.1f}±{np.std(wars):.1f}%, UAR={np.mean(uars):.1f}±{np.std(uars):.1f}%")
 
# Paired Δ_A across seeds
print("\n--- Δ_A across seeds ---")
for i, seed in enumerate(SEEDS):
    da = multi_results['M3'][i]['uar'] - multi_results['M1'][i]['uar']
    print(f"  Seed {seed}: Δ_A = {da:+.1f} pts")
 
da_values = [multi_results['M3'][i]['uar'] - multi_results['M1'][i]['uar'] for i in range(len(SEEDS))]
print(f"  Mean Δ_A = {np.mean(da_values):+.1f} ± {np.std(da_values):.1f} pts")
 
# Bootstrap CI on best seed
best_seed_idx = np.argmax([r['uar'] for r in multi_results['M3']])
m1_preds = multi_results['M1'][best_seed_idx]['preds']
m3_preds = multi_results['M3'][best_seed_idx]['preds']
labels = multi_results['M3'][best_seed_idx]['labels']
 
mean_da, lo_da, hi_da, p_pos = bootstrap_delta_ci(m1_preds, m3_preds, labels)
print(f"\n--- Bootstrap CI (best seed, 1000 resamples) ---")
print(f"  Δ_A = {mean_da:+.1f}% [{lo_da:+.1f}, {hi_da:+.1f}] (95% CI)")
print(f"  P(Δ_A > 0) = {p_pos:.3f}")
sig = "SIGNIFICANT" if lo_da > 0 else "NOT significant"
print(f"  → {sig} (CI {'excludes' if lo_da > 0 else 'includes'} zero)")

EXP 1: Multiple Seeds (3 seeds × M1, M3)

--- Seed 42 ---
  M1 (context only)... 

/tmp/ipykernel_58/1406243742.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)


WAR=36.4%, UAR=23.1%
  M3 (full)... 

/tmp/ipykernel_58/1406243742.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)
/tmp/ipykernel_58/1406243742.py:114: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


WAR=38.3%, UAR=26.2%

--- Seed 123 ---
  M1 (context only)... WAR=37.3%, UAR=24.9%
  M3 (full)... WAR=36.6%, UAR=25.5%

--- Seed 456 ---
  M1 (context only)... WAR=28.6%, UAR=22.2%
  M3 (full)... WAR=35.0%, UAR=23.0%

--- Multi-seed summary ---
M1: WAR=34.1±3.9%, UAR=23.4±1.1%
M3: WAR=36.6±1.3%, UAR=24.9±1.4%

--- Δ_A across seeds ---
  Seed 42: Δ_A = +3.1 pts
  Seed 123: Δ_A = +0.6 pts
  Seed 456: Δ_A = +0.8 pts
  Mean Δ_A = +1.5 ± 1.1 pts

--- Bootstrap CI (best seed, 1000 resamples) ---
  Δ_A = +3.1% [-0.3, +6.4] (95% CI)
  P(Δ_A > 0) = 0.965
  → NOT significant (CI includes zero)


In [6]:
print("\n" + "=" * 70)
print("EXP 2: Shuffled-A Control")
print("Is M3's improvement from A-B relationship or just extra parameters?")
print("=" * 70)
 
# Use best seed
best_seed = SEEDS[best_seed_idx]
torch.manual_seed(best_seed); torch.cuda.manual_seed_all(best_seed)
 
# Shuffled train/val/test loaders
trl_shuf = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, train_idx, shuffle_a=True, seed=best_seed),
                       batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_shuf = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, val_idx, shuffle_a=True, seed=best_seed+1),
                       batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
tst_shuf = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, test_idx, shuffle_a=True, seed=best_seed+2),
                       batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
 
# Normal test loader for evaluation (we want to compare on SAME test samples)
tst_normal = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, test_idx),
                         batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
 
print("  M3_shuffled (train with random A, test with random A)...", end=" ", flush=True)
m3_shuf = M3_Full().to(DEVICE)
r_shuf = train_and_eval(m3_shuf, trl_shuf, val_shuf, tst_shuf, N_EPOCHS, name="m3_shuffled")
print(f"WAR={r_shuf['war']:.1f}%, UAR={r_shuf['uar']:.1f}%")
 
# Also test shuffled model on normal (unshuffled) test data
print("  M3_shuffled tested on normal data...", end=" ", flush=True)
m3_shuf.eval(); tp,tl=[],[]
with torch.no_grad():
    for b in tst_normal:
        bg = {k:v.to(DEVICE) for k,v in b.items()}
        tp.extend(m3_shuf(bg).argmax(-1).cpu().numpy()); tl.extend(b['target'].numpy())
w,u = compute_metrics(tp,tl)
print(f"WAR={w:.1f}%, UAR={u:.1f}%")
 
# Compare
best_m1 = multi_results['M1'][best_seed_idx]
best_m3 = multi_results['M3'][best_seed_idx]
 
print(f"\n--- Comparison ---")
print(f"  M1 (context only):     UAR={best_m1['uar']:.1f}%")
print(f"  M3_shuffled (random A): UAR={r_shuf['uar']:.1f}%")
print(f"  M3 (true A):            UAR={best_m3['uar']:.1f}%")
print(f"\n  M3_shuffled - M1 = {r_shuf['uar'] - best_m1['uar']:+.1f} pts (extra params effect)")
print(f"  M3 - M3_shuffled = {best_m3['uar'] - r_shuf['uar']:+.1f} pts (genuine A-B relationship)")
 
if best_m3['uar'] > r_shuf['uar'] + 1:
    print(f"\n  ✓ True A outperforms shuffled A → improvement comes from A-B relationship, not parameters")
else:
    print(f"\n  ✗ True A ≈ shuffled A → improvement may come from extra parameters, not A-B relationship")


EXP 2: Shuffled-A Control
Is M3's improvement from A-B relationship or just extra parameters?
  M3_shuffled (train with random A, test with random A)... 

/tmp/ipykernel_58/1406243742.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)
/tmp/ipykernel_58/1406243742.py:114: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


WAR=39.9%, UAR=26.9%
  M3_shuffled tested on normal data... WAR=40.6%, UAR=27.3%

--- Comparison ---
  M1 (context only):     UAR=23.1%
  M3_shuffled (random A): UAR=26.9%
  M3 (true A):            UAR=26.2%

  M3_shuffled - M1 = +3.8 pts (extra params effect)
  M3 - M3_shuffled = -0.7 pts (genuine A-B relationship)

  ✗ True A ≈ shuffled A → improvement may come from extra parameters, not A-B relationship


In [7]:
print("\n" + "=" * 70)
print("EXP 3: Counterfactual — Replace A with same-emotion different person")
print("Does B need A's SPECIFIC behavior or just A's emotion LABEL?")
print("=" * 70)
 
# Use best M3 model (already trained)
m3_best = M3_Full().to(DEVICE)
m3_best.load_state_dict(torch.load(f'/kaggle/working/m3_s{best_seed}.pt', map_location=DEVICE, weights_only=True))
m3_best.eval()
 
# Group clip3 features by emotion label
emo_to_clips = defaultdict(list)
for idx in train_idx:
    e = mcis_index[idx]
    emo = e['clip3_emotion']
    if emo >= 0:
        emo_to_clips[emo].append(e['feature_files'][2])
 
print("  Emotion → available clip3 replacements:")
for emo in range(7):
    print(f"    {emo_names[emo]}: {len(emo_to_clips[emo])} clips")
 
# Test with counterfactual A
rng_cf = random.Random(42)
tp_cf, tl_cf = [], []
tp_orig, tl_orig = [], []
 
for batch in tst_normal:
    bg = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    
    # Original prediction
    with torch.no_grad():
        logits_orig = m3_best(bg)
        tp_orig.extend(logits_orig.argmax(-1).cpu().numpy())
        tl_orig.extend(batch['target'].numpy())
    
    # Counterfactual: replace clip3 features with same-emotion different person
    bg_cf = {k: v.clone() if isinstance(v, torch.Tensor) else v for k, v in bg.items()}
    
    for i in range(bg['target'].size(0)):
        emo = batch['clip3_emotion'][i].item()
        if emo >= 0 and len(emo_to_clips[emo]) > 0:
            # Pick random replacement with same emotion
            replacement_file = rng_cf.choice(emo_to_clips[emo])
            d = torch.load(os.path.join(FEATURES_DIR, replacement_file), map_location='cpu', weights_only=False)
            bg_cf['clip3_face'][i] = d['face_features'].to(DEVICE)
            bg_cf['clip3_ori'][i] = d['ori_features'].to(DEVICE)
            bg_cf['clip3_text'][i] = d['text_feature'].to(DEVICE)
            bg_cf['clip3_audio'][i] = d.get('audio_feature', torch.zeros(527)).to(DEVICE)
    
    with torch.no_grad():
        logits_cf = m3_best(bg_cf)
        tp_cf.extend(logits_cf.argmax(-1).cpu().numpy())
        tl_cf.extend(batch['target'].numpy())
 
tp_orig, tl_orig = np.array(tp_orig), np.array(tl_orig)
tp_cf, tl_cf = np.array(tp_cf), np.array(tl_cf)
 
w_orig, u_orig = compute_metrics(tp_orig, tl_orig)
w_cf, u_cf = compute_metrics(tp_cf, tl_cf)
 
print(f"\n--- Results ---")
print(f"  M3 (original A):        WAR={w_orig:.1f}%, UAR={u_orig:.1f}%")
print(f"  M3 (same-label swap A): WAR={w_cf:.1f}%, UAR={u_cf:.1f}%")
print(f"  Drop: {u_cf - u_orig:+.1f} pts")
 
if u_orig > u_cf + 1:
    print(f"\n  ✓ Original A > swapped A → fine-grained behavior matters, not just label")
    print(f"  B needs to know HOW A is angry, not just THAT A is angry")
elif abs(u_orig - u_cf) < 1:
    print(f"\n  ~ Original A ≈ swapped A → emotion label captures most of A's signal")
    print(f"  Fine-grained behavior adds little beyond the category")
else:
    print(f"\n  ✗ Swapped A > original A → unexpected, may indicate noise in clip matching")
 
# Bootstrap significance
mean_d, lo_d, hi_d, p = bootstrap_delta_ci(tp_cf, tp_orig, tl_orig)
print(f"  Bootstrap Δ = {mean_d:+.1f}% [{lo_d:+.1f}, {hi_d:+.1f}] (95% CI)")


EXP 3: Counterfactual — Replace A with same-emotion different person
Does B need A's SPECIFIC behavior or just A's emotion LABEL?


/tmp/ipykernel_58/1406243742.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)
/tmp/ipykernel_58/1406243742.py:114: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.trans = nn.TransformerEncoder(el, num_layers=2)


  Emotion → available clip3 replacements:
    angry: 423 clips
    disgust: 162 clips
    fear: 25 clips
    happy: 441 clips
    neutral: 451 clips
    sad: 270 clips
    surprise: 208 clips

--- Results ---
  M3 (original A):        WAR=38.3%, UAR=26.2%
  M3 (same-label swap A): WAR=35.7%, UAR=23.8%
  Drop: -2.4 pts

  ✓ Original A > swapped A → fine-grained behavior matters, not just label
  B needs to know HOW A is angry, not just THAT A is angry
  Bootstrap Δ = +2.4% [-0.0, +5.0] (95% CI)


In [8]:
print("\n" + "=" * 70)
print("EXP 4: Label-only A vs Full-features A")
print("P(B|C,E_A) vs P(B|C,X_A)")
print("=" * 70)
 
torch.manual_seed(best_seed); torch.cuda.manual_seed_all(best_seed)
 
trl = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, train_idx, seed=best_seed),
                 batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val = DataLoader(HiEFDataset(mcis_index, FEATURES_DIR, val_idx),
                 batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
 
print("  M_label: P(B|C, E_A) — context + 7-class label...", end=" ", flush=True)
m_label = M_EmoLabel().to(DEVICE)
print(f"Params: {sum(p.numel() for p in m_label.parameters()):,}")
r_label = train_and_eval(m_label, trl, val, tst_normal, N_EPOCHS, name="m_label")
print(f"  WAR={r_label['war']:.1f}%, UAR={r_label['uar']:.1f}%")
 
print(f"\n--- Comparison ---")
print(f"  M1: P(B|C)          UAR={best_m1['uar']:.1f}%")
print(f"  M_label: P(B|C,E_A) UAR={r_label['uar']:.1f}%  (label adds {r_label['uar']-best_m1['uar']:+.1f})")
print(f"  M3: P(B|C,X_A)      UAR={best_m3['uar']:.1f}%  (features add {best_m3['uar']-best_m1['uar']:+.1f})")
print(f"\n  Features vs label: {best_m3['uar']-r_label['uar']:+.1f} pts")
 
if best_m3['uar'] > r_label['uar'] + 2:
    print(f"  ✓ Full features >> label → B needs HOW A is angry, not just THAT A is angry")
elif r_label['uar'] > best_m1['uar'] + 2:
    print(f"  ~ Label already helps → emotion category carries the main signal")
    print(f"  Full features add modest extra: {best_m3['uar']-r_label['uar']:+.1f} pts")
else:
    print(f"  ✗ Neither label nor features help much → A's contribution is weak overall")


EXP 4: Label-only A vs Full-features A
P(B|C,E_A) vs P(B|C,X_A)
  M_label: P(B|C, E_A) — context + 7-class label... Params: 18,292,231


/tmp/ipykernel_58/1406243742.py:56: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.t = nn.TransformerEncoder(el, num_layers=2)


  WAR=38.5%, UAR=25.9%

--- Comparison ---
  M1: P(B|C)          UAR=23.1%
  M_label: P(B|C,E_A) UAR=25.9%  (label adds +2.8)
  M3: P(B|C,X_A)      UAR=26.2%  (features add +3.1)

  Features vs label: +0.3 pts
  ~ Label already helps → emotion category carries the main signal
  Full features add modest extra: +0.3 pts


In [9]:
print("\n" + "=" * 70)
print("FULL SUMMARY")
print("=" * 70)
 
print(f"""
1. MULTI-SEED STABILITY:
   Δ_A = {np.mean(da_values):+.1f} ± {np.std(da_values):.1f} pts across {len(SEEDS)} seeds
   Bootstrap 95% CI: [{lo_da:+.1f}, {hi_da:+.1f}]
 
2. SHUFFLED-A CONTROL:
   M3 (true A):     UAR={best_m3['uar']:.1f}%
   M3 (shuffled A): UAR={r_shuf['uar']:.1f}%
   M1 (no A):       UAR={best_m1['uar']:.1f}%
   → Genuine A-B effect: {best_m3['uar'] - r_shuf['uar']:+.1f} pts
   → Extra params effect: {r_shuf['uar'] - best_m1['uar']:+.1f} pts
 
3. COUNTERFACTUAL:
   Original A:      UAR={u_orig:.1f}%
   Same-label swap:  UAR={u_cf:.1f}%
   → Fine-grained behavior adds: {u_orig - u_cf:+.1f} pts beyond label
 
4. LABEL vs FEATURES:
   P(B|C):          UAR={best_m1['uar']:.1f}%
   P(B|C, E_A):     UAR={r_label['uar']:.1f}%  (label value: {r_label['uar']-best_m1['uar']:+.1f})
   P(B|C, X_A):     UAR={best_m3['uar']:.1f}%  (feature value: {best_m3['uar']-best_m1['uar']:+.1f})
""")
 
# Save
save_data = {
    'multi_seed': {
        'seeds': SEEDS,
        'M1_uars': [r['uar'] for r in multi_results['M1']],
        'M3_uars': [r['uar'] for r in multi_results['M3']],
        'delta_a_mean': float(np.mean(da_values)),
        'delta_a_std': float(np.std(da_values)),
        'bootstrap_ci': [float(lo_da), float(hi_da)],
    },
    'shuffled_a': {
        'm1_uar': best_m1['uar'], 'm3_uar': best_m3['uar'], 'shuffled_uar': r_shuf['uar'],
        'genuine_effect': best_m3['uar'] - r_shuf['uar'],
        'param_effect': r_shuf['uar'] - best_m1['uar'],
    },
    'counterfactual': {
        'original_uar': float(u_orig), 'swapped_uar': float(u_cf),
        'delta': float(u_orig - u_cf),
    },
    'label_vs_features': {
        'm1_uar': best_m1['uar'], 'label_uar': r_label['uar'], 'm3_uar': best_m3['uar'],
    },
}
with open('/kaggle/working/rigorous_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)
print("Saved to /kaggle/working/rigorous_results.json")


FULL SUMMARY

1. MULTI-SEED STABILITY:
   Δ_A = +1.5 ± 1.1 pts across 3 seeds
   Bootstrap 95% CI: [-0.3, +6.4]
 
2. SHUFFLED-A CONTROL:
   M3 (true A):     UAR=26.2%
   M3 (shuffled A): UAR=26.9%
   M1 (no A):       UAR=23.1%
   → Genuine A-B effect: -0.7 pts
   → Extra params effect: +3.8 pts
 
3. COUNTERFACTUAL:
   Original A:      UAR=26.2%
   Same-label swap:  UAR=23.8%
   → Fine-grained behavior adds: +2.4 pts beyond label
 
4. LABEL vs FEATURES:
   P(B|C):          UAR=23.1%
   P(B|C, E_A):     UAR=25.9%  (label value: +2.8)
   P(B|C, X_A):     UAR=26.2%  (feature value: +3.1)

Saved to /kaggle/working/rigorous_results.json
